# 装甲板 YOLO26-OBB 训练（Colab GPU）

在 Google Colab 免费 GPU 上训练 `deep_learning` 方案。

**准备**：菜单 `代码执行程序 → 更改运行时类型 → 硬件加速器 = T4 GPU`，然后从上到下依次运行。

流程：检查 GPU → 克隆仓库 → 安装依赖 → 准备 dataset2 → 构建增广数据集 → 训练 → 查看/下载结果。

In [ ]:
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
REPO = 'https://github.com/diyanqi/rm_armor.git'
!git clone $REPO rm_armor 2>/dev/null || (cd rm_armor && git pull)
%cd rm_armor

In [ ]:
# Colab 已自带 torch/cv2/numpy，只补装 ultralytics
!pip -q install 'ultralytics>=8.4'

## 准备数据集

把 `dataset1` + `dataset2`（共 1790 张图 + labels）打包成 `datasets.zip`，放到 Google Drive 的 `MyDrive` 根目录。
如果不想用 Drive，也可以在上传单元格里直接选文件。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 方式 A：从 Drive 解压（把路径改成你的 zip 位置）
!cp /content/drive/MyDrive/datasets.zip . && unzip -q -o datasets.zip

# 方式 B：直接上传（取消下面两行注释，运行后选择 datasets.zip）
# from google.colab import files; files.upload()
# !unzip -q -o datasets.zip

!echo dataset1: $(ls dataset1/images | wc -l)   dataset2: $(ls dataset2/images | wc -l)

## 构建增广数据集

合并 dataset1 + dataset2，标签转成 24 类（颜色×车型）+ 近重复隔离划分 + 离线增广（噪点/遮挡/亮度）到约 4 倍。

In [ ]:
!python deep_learning/apps/build_dataset.py --format jpg --clean

## 训练

Colab T4 显存 16G，`--batch 16`；跑不动就降到 8。
建议把结果目录指向 Drive，避免运行时段断开后产物丢失：`--project /content/drive/MyDrive/rm_runs`。

In [ ]:
!python deep_learning/apps/train.py \
  --model yolo26n-obb.pt \
  --epochs 150 --imgsz 1024 --batch 16 --device 0 --workers 2 \
  --project /content/drive/MyDrive/rm_runs --name colab_yolo26n

## 查看训练结果

In [ ]:
from IPython.display import Image, display
import glob, os
png = sorted(glob.glob('/content/drive/MyDrive/rm_runs/*/results.png'), key=os.path.getmtime)[-1]
print(png)
display(Image(png))

## 推理 / 下载结果

In [ ]:
!python deep_learning/apps/infer.py --source dataset2/images --limit 50 --device 0 --out preview

# 打包 best.pt 与可视化结果下载
!zip -qr results.zip /content/drive/MyDrive/rm_runs preview
from google.colab import files
files.download('results.zip')